# Somatic Variant Caller

A helper ipynb notebook to setup config file and bash scripts to run individual sample (tumor normal or tumor only) using the ChapuyLab somatic_variant_caller pipeline

We used slurm to run our pipeline so this notebook only contains the setup for that please feel free to modify for other engines using the snakemake --profile command below

This pipeline has been tested on hg19

Start by setting up of basic_paths

In [ ]:
import os
from pathlib import Path
import pandas as pd
import glob

##############################################################################
################ Update this for your own cluster architecture ###############
##############################################################################

base_path = Path("/base_path") 
baits_file = "hg19/Agilent_SureSelectv2_baits.bed"

# download the below files from here 
# https://console.cloud.google.com/storage/browser/gatk-best-practices/somatic-b37?pageState=(%22StorageObjectListTable%22:(%22f%22:%22%255B%255D%22))
germline = "hg19/af-only-gnomad.raw.sites.vcf"
genome = 'hg19/hs37d5_PhiX.fa' # you can use the hg19 build provided by broad
common_biallelic = 'hg19/small_exac_common_3.vcf'
pon = "hg19/pon.vcf"
genome_build = "hg19"

path_to_annovar= "/please/insert/path/to/annovar" # download annovar and add its path here
annovar_db = "/please/insert/path/to/annovar_humandb" # download annovar humandb and add its path here

# https://github.com/naumanjaved/fingerprint_maps/tree/master/map_files
haplotypemap = "hg19/hg19_nochr.map"
pkl_detin = "hg19/exac.pickle_high_af" # downloadable from https://github.com/getzlab/deTiN/tree/master/example_data


path_to_activate = "miniforge3/bin/activate" # please set the path to where your activate function is for your conda environment

workflow_profile = "profile/config.yaml" # please set path to the workflow profile you want to use I am providing one for example conda_prefix, singularity_prefix are required
snakmake_slurm_profile="slurm_profile" # this is very important to change for your own cluster

##############################################################################
# Can remain same upto user to decide I personally like the folder structure #
##############################################################################


output_folder = base_path / "results" / "view_by_pid"
scripts_folder = base_path / "processing_scripts" / "mutect2" / "array_scripts"
log_folder = base_path / "processing_scripts" / "mutect2" / "array_logs"


os.makedirs(scripts_folder, exist_ok=True)
os.makedirs(log_folder, exist_ok=True)

## Metadata

Create a bam metadata df which can be create automatically if the alignment or realignment pipeline from chapuyLab  was used 

If not please provide a metadata sheet in TSV format which contains the following information as columns
1. BAM_FILE: Path to bam file
2. SAMPLE_TYPE: control / tumor (can be anything as long for control/normal samples the word normal or control exist in the entry)
3. PATIENT_ID: Unique identifier for each Patient two patient can have same sample type 
4. SAMPLE_NAME: Generally its combination of PATIENT_ID and SAMPLE_TYPE joined by "_" can  be left empty as will be overwritten (this needs to be unique)

Please don't use '_' in SAMPLE_TYPE or PATIENT_ID this is very important for downstream processing

In [ ]:
# If using the alignment or realignment pipeline from chapuyLab 

create_metadata=True
bam_meta_file_path=""
if create_metadata:
    bam=glob.glob(base_path / "results" / "view-by-pid" / "**"/ "*.cram", recursive=True)
    new_df=pd.DataFrame(columns=['BAM_FILE', 'SAMPLE_NAME', 'SAMPLE_TYPE', 'PATIENT_ID'])
    new_df['BAM_FILE']=bam
    new_df['SAMPLE_TYPE']=new_df['BAM_FILE'].apply(lambda x: x.split('/')[-4])
    new_df['PATIENT_ID']=new_df['BAM_FILE'].apply(lambda x: x.split('/')[-5])
    new_df['SAMPLE_NAME']=new_df['PATIENT_ID']+'_'+new_df['SAMPLE_TYPE']
    new_df['INSERT_SIZE'] = new_df.apply(lambda x: '/'.join(x['BAM_FILE'].split('/')[:-2]) +'/metrics/'+x['SAMPLE_NAME']+'_insert_size_metrics.txt', axis=1)
else:
    new_df=pd.read_table(bam_meta_file_path)
    new_df['SAMPLE_TYPE']=new_df['SAMPLE_TYPE'].str.replace("_", "-")
    new_df['PATIENT_ID']=new_df['PATIENT_ID'].str.replace("_", "-")
    new_df['SAMPLE_NAME']=new_df['PATIENT_ID']+'_'+new_df['SAMPLE_TYPE']

assert new_df.SAMPLE_NAME.is_unique , f"Duplicate sample names found: {new_df['SAMPLE_NAME'].duplicated().sum()} duplicates"

In [ ]:
# creating a normal and tumor dataframe for convinience 


normal_df = new_df[(new_df['SAMPLE_TYPE'].str.contains('normal')) | (new_df['SAMPLE_TYPE'].str.contains('control'))]
normal_df.set_index('PATIENT_ID', inplace=True)
tumor_df = new_df[~new_df['SAMPLE_NAME'].isin(normal_df.SAMPLE_NAME)]

## GetSampeName

Get the rgid for each of the bam files and store them locally

This will create a script for each of the samples (numbered 1 to len(new_df)) in the scripts folder please execute them one by one or submiting them as job array

In [ ]:
sample_name_path=base_path / "results" / "metadata" / "sample_names")

os.makedirs(workdir, exist_ok=True)
os.makedirs(bash_folder, exist_ok=True)



count=0
for index, row in new_df.iterrows():
    count+=1
    with open(scripts_folder / (str(count)+'_sample_names.sh'), 'w') as handle:
        handle.write(f'source {path_to_activate} base_env && '+
                     'gatk GetSampleName -R '+genome+' -I '+ row.BAM_FILE + ' '+
                     '-O '+str(sample_name_path / str(row.SAMPLE_NAME+'.txt'))
                    )

## Create scripts

This will create a scripts and config file for each of the samples (numbered 1 to len(tumor_df)) in the scripts folder please execute them one by one or submiting them as job array

In [ ]:
import yaml
from pathlib import Path

count=0
count_skip=0
skipped_pids=list()

yaml_dict = yaml.safe_load(Path("config/config_base.yaml").read_text())
yaml_dict['genome']=genome
yaml_dict['common_biallelic'] = common_biallelic
yaml_dict['germline'] = germline 
yaml_dict['haplotypemap']=haplotypemap 

yaml_dict['pon']=pon
yaml_dict['pkl_detin']=pkl_detin

yaml_dict['run_annovar']=True
yaml_dict['path_to_annovar']=path_to_annovar
yaml_dict['annovar_db']=annovar_db

yaml_dict["interval_folder"]= ""# only needed for wgs data

yaml_dict['stepper']='all'
yaml_dict['scatter_count']=2 # how many mutect processes ot spawn helpful when running with WGS else leave at 2

fakenormal_pid=None # provide a fake normal pid for samples without a normal if possible else will be run without

def getRGid(sample_name, sample_name_path):
    with open(sample_name_path / (sample_name+'.txt')) as handle:
        return handle.readline().strip()

def safeDelete(entry, yaml_dict):
    if entry in yaml_dict:
        del yaml_dict[entry]
    return yaml_dict

    
for index, row in tumor_df.iterrows():
    yaml_dict["fake_normal"]=False
    yaml_dict['target_file']=baits_file

    if row.PATIENT_ID in set(normal_df.index) or fakenormal_pid is not None:
        norm_pid=row.PATIENT_ID
        if fakenormal_pid:
            print('Normal not found for patient'+ row['PATIENT_ID'])
            print('Using fake normal mode')
            norm_pid=fakenormal_pid
            yaml_dict["fake_normal"]=True
            
        yaml_dict['normal_sample']=normal_df.loc[norm_pid, 'SAMPLE_TYPE']
        yaml_dict["normal_bam_file"]=normal_df.loc[norm_pid, 'BAM_FILE']
        yaml_dict["rg_normal"]=getRGid(normal_df.loc[norm_pid, 'SAMPLE_NAME'], sample_name_path)
    else:
        yaml_dict=safedelete("normal_sample" , yaml_dict)
        yaml_dict=safedelete("normal_bam_file" , yaml_dict)
        yaml_dict=safedelete("rg_normal" , yaml_dict)
        
        
    tumour_bam_file = row.BAM_FILE
    rg_tumor_id = getRGid(row.SAMPLE_NAME)
    
    yaml_dict['insertSize']=row['INSERT_SIZE']
    yaml_dict['stepper']='all'
   
    yaml_dict['workdir']=output_folder / row.PATIENT_ID / "somatic_mutations" / row.SAMPLE_NAME
    
    os.makedirs(yaml_dict['workdir'], exist_ok=True)
    
    yaml_dict['logdir']=yaml_dict['workdir'] / 'logs'
    
    os.makedirs(yaml_dict['logdir'], exist_ok=True)
    
    yaml_dict['pid']=row.PATIENT_ID
    yaml_dict['tumour_sample']=row.SAMPLE_TYPE
    
    yaml_dict['rg_tumour']= getRGid(row.SAMPLE_NAME)
    
    config_file = yaml_dict['workdir'] / (row.SAMPLE_NAME+'.yaml')

    
    yaml_dict['workdir']=str(yaml_dict['workdir'])
    yaml_dict['logdir']=str(yaml_dict['logdir'])
    yaml_dict['tumour_bam_file'] = row.BAM_FILE
    yaml_dict["filter_mutect_extra"] = "--max-events-in-region 10" # specific for lymphoma
    
    with open(config_file, 'w') as ff:
        yaml.dump(yaml_dict, ff)

        
    cmd=(
        f"module purge;source {path_to_activate} base_env && "+
        "cd "+yaml_dict['workdir'] +" "+
        "&& snakemake "+
        f"--profile {snakmake_slurm_profile} "+ # delete this line if not using slurm
        f"--workflow-profile {workflow_profile} "+ # edit this file for slurm dependent modifications
        "--use-singularity --use-conda -j 10 "+ # it is highly recommended to use singularity profile with mamba
        "--conda-frontend mamba "+
        "--configfile "+str(config_file)+" "+
        "--snakefile workflow/Snakefile"
    )
    count+=1
    with open(bash_folder / (str(count)+'.sh'), 'w') as handle:
        handle.write(cmd + " --unlock && "+cmd) # the unlock has been added to deal with unwanted interupption can be safely removed 
print('Total Patients %s' %(len(new_df['PATIENT_ID'].unique())))
print(skipped_pids)
print('Skipped Patients %s' %(count_skip))

In [ ]:
count